# ResNet 잔차 신경망 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/slides/v2/ml1/week10/resnet/lab/resnet.ipynb)

입력을 건너뛰어 더하는 지름길 $\mathbf{y} = F(\mathbf{x}) + \mathbf{x}$.

- **파트 1**: 잔차 블록 직접 만들기
- **파트 2**: 손글씨 1만 장: 일반 vs 잔차, 6 · 18 · 34층 (몇 분 걸린다)
- **파트 3**: 층별 기울기 크기 + torchvision `resnet18` 구조

`None` 으로 비워 둔 `TODO` 줄을 채우고 셀을 차례로 실행하세요.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import datasets

## 파트 1. 잔차 블록

In [ ]:
class Block(nn.Module):
    def __init__(self, width, residual):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))
        self.residual = residual

    def forward(self, x):
        if self.residual:
            return None  # TODO: F(x) + x, then ReLU
        return torch.relu(self.f(x))

def make_net(blocks, residual, width=64):
    return nn.Sequential(nn.Linear(784, width), nn.ReLU(), *[Block(width, residual) for _ in range(blocks)], nn.Linear(width, 10))

b = Block(4, residual=True)
nn.init.zeros_(b.f[2].weight); nn.init.zeros_(b.f[2].bias)  # make F(x) = 0
x = torch.tensor([[1.0, 2.0, 0.5, 3.0]])
print("input :", x.tolist()); print("output:", b(x).tolist(), " <- passes straight through when F = 0")

**예상 결과**

```
input : [[1.0, 2.0, 0.5, 3.0]]
output: [[1.0, 2.0, 0.5, 3.0]]  <- passes straight through when F = 0
```

## 파트 2. 깊이를 늘리면

블록 2 · 8 · 16개 = 층 6 · 18 · 34개. Adam (lr 1e-3), 배치 64, 5 에폭, 학습 정확도를 잰다.

In [ ]:
tr = datasets.MNIST("data", train=True, download=True)
idx = torch.randperm(60000, generator=torch.Generator().manual_seed(0))[:10000]
X = tr.data[idx].reshape(-1, 784).float() / 255; y = tr.targets[idx]

def train(net, X, y, epochs=5):
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    for ep in range(epochs):
        for b in torch.randperm(len(y), generator=torch.Generator().manual_seed(ep)).split(64):
            opt.zero_grad(); nn.functional.cross_entropy(net(X[b]), y[b]).backward(); opt.step()
    with torch.no_grad():
        return (net(X).argmax(1) == y).float().mean().item()

results = {}
for blocks in [2, 8, 16]:
    for residual in [False, True]:
        torch.manual_seed(0)
        results[(blocks, residual)] = None  # TODO: train and get training accuracy
        print(f"{2 * blocks + 2:2d} layers, residual={residual}: train accuracy {results[(blocks, residual)]:.3f}")

**예상 결과**

```
 6 layers, residual=False: train accuracy 0.937
 6 layers, residual=True: train accuracy 0.958
18 layers, residual=False: train accuracy 0.574
18 layers, residual=True: train accuracy 0.951
34 layers, residual=False: train accuracy 0.117
34 layers, residual=True: train accuracy 0.962
```

## 파트 3. 층별 기울기와 실제 ResNet

In [ ]:
def first_layer_grad(net, X, y):
    net.zero_grad(); nn.functional.cross_entropy(net(X), y).backward()
    return net[0].weight.grad.abs().mean().item()

for residual in [False, True]:
    torch.manual_seed(0)
    print(f"34 layers, residual={residual}: |grad| of the first layer = {first_layer_grad(make_net(16, residual), X[:512], y[:512]):.2e}")

from torchvision.models import resnet18
r = resnet18(weights=None)
print(r.layer2[0])                                        # first block of stage 2: note the 'downsample' shortcut (1x1 conv)
print("total parameters:", f"{sum(p.numel() for p in r.parameters()):,}")

**예상 결과**

```
34 layers, residual=False: |grad| of the first layer = 9.66e-17
34 layers, residual=True: |grad| of the first layer = 7.22e-04
BasicBlock(
  (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (downsample): Sequential(
    (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
)
total parameters: 11,689,512
```

**확인 질문**
1. 파트 1 에서 F 가 0 이면 출력이 입력과 같다. 일반 블록이었다면 같은 입력에 무엇이 나왔을까?
2. 파트 2 에서 일반 신경망 34층의 학습 정확도는? 과대적합인가?
3. 파트 3 에서 첫 층 기울기 크기는 몇 배 차이 나나? 지난 시간의 기울기 소실과 어떻게 연결되나?
4. `resnet18` 의 `downsample` 은 왜 있을까? (힌트: 채널 64 → 128, 크기 절반)

데이터 출처: LeCun, Cortes, Burges, MNIST (torchvision) · He et al. (2015) arXiv:1512.03385.